# yt-dlp Heroku Deployer

## Features
- Deploy directly from GitHub to Heroku using the Builds API (no GitHub OAuth required).
- Full support for Heroku Teams.
- Configure and manage Config Vars, including `UPSTREAM_REPO` and `UPSTREAM_BRANCH`.
- Manage dyno scaling and tiers.
- Integrated log streaming and release management.
- Configuration backup and restore.

## Requirements
- A Heroku API Key (available in Heroku Account Settings).
- A valid GitHub repository containing your yt-dlp bot code.

## Deployment Workflow
1. Authenticate with Heroku.
2. Select a Team and configure App details.
3. Set Config Vars and manage Buildpacks.
4. Deploy code from GitHub via Tarball.
5. Manage Dynos and Monitor Logs.

In [ ]:
# @title 1. Install Dependencies
!pip install -q requests tqdm

In [ ]:
# @title 2. Authentication
# @markdown Enter your Heroku API Key to authenticate. You can find this in your Heroku Account Settings.
HEROKU_API_KEY = "" # @param {type:"string"}

import requests

headers = {
    "Accept": "application/vnd.heroku+json; version=3",
    "Authorization": f"Bearer {HEROKU_API_KEY}"
}

def check_auth():
    if not HEROKU_API_KEY:
        print("❌ Please provide a Heroku API Key.")
        return None
    
    print("Checking authentication...")
    resp = requests.get("https://api.heroku.com/account", headers=headers)
    
    if resp.status_code == 200:
        data = resp.json()
        print("✅ Authenticated successfully!")
        print(f"👤 User: {data.get('name', 'N/A')}")
        print(f"📧 Email: {data.get('email')}")
        return data
    else:
        print(f"❌ Authentication failed: {resp.status_code} - {resp.text}")
        return None

user_info = check_auth()

In [ ]:
# @title 3. Team Discovery
# @markdown Run this cell to fetch and select a Heroku Team (or Personal account).
import json

def get_teams():
    print("Fetching available teams...")
    resp = requests.get("https://api.heroku.com/teams", headers=headers)
    if resp.status_code == 200:
        teams = resp.json()
        print("Available Teams:")
        print("0. Personal Account")
        for i, team in enumerate(teams, 1):
            print(f"{i}. {team['name']} (Role: {team.get('role', 'N/A')})")
        return teams
    else:
        print(f"❌ Failed to fetch teams: {resp.text}")
        return []

teams = get_teams()


In [ ]:
# @title Select Team
# @markdown Enter the number corresponding to your choice from the list above.
TEAM_CHOICE = 0 # @param {type:"integer"}

SELECTED_TEAM = None
if TEAM_CHOICE == 0:
    print("✅ Selected: Personal Account")
elif 1 <= TEAM_CHOICE <= len(teams):
    SELECTED_TEAM = teams[TEAM_CHOICE - 1]['name']
    print(f"✅ Selected Team: {SELECTED_TEAM}")
else:
    print("❌ Invalid selection.")


In [ ]:
# @title 4. Application Management
# @markdown Provide an App Name and Region.
APP_NAME = "" # @param {type:"string"}
REGION = "us" # @param ["us", "eu"]

def get_app(app_name):
    resp = requests.get(f"https://api.heroku.com/apps/{app_name}", headers=headers)
    return resp.json() if resp.status_code == 200 else None

def create_app(app_name, region, team):
    payload = {"region": region}
    if app_name:
        payload["name"] = app_name
        
    url = "https://api.heroku.com/teams/apps" if team else "https://api.heroku.com/apps"
    if team:
        payload["team"] = team
        
    print(f"Creating app{' in team ' + team if team else ''}...")
    resp = requests.post(url, headers=headers, json=payload)
    if resp.status_code in [201, 200]:
        app_data = resp.json()
        print(f"✅ App created successfully: {app_data['name']}")
        return app_data
    else:
        print(f"❌ Failed to create app: {resp.text}")
        return None

def manage_app():
    if not APP_NAME:
        print("❌ Please provide an APP_NAME.")
        return None
        
    app_data = get_app(APP_NAME)
    if app_data:
        print(f"✅ App '{APP_NAME}' already exists. Reusing it.")
        return app_data
    else:
        print(f"App '{APP_NAME}' not found. Attempting to create...")
        return create_app(APP_NAME, REGION, SELECTED_TEAM)

app_info = manage_app()
if app_info:
    APP_NAME = app_info['name'] # Update in case it was blank and auto-generated


In [ ]:
# @title 5. Config Vars
# @markdown Fill in the dictionary below. Empty values (`""`) will be ignored, meaning they won't overwrite existing variables on Heroku. Existing variables not listed here will be preserved.

CONFIG_VARS = {
    "BOT_TOKEN": "",
    "API_ID": "",
    "API_HASH": "",
    "DATABASE_URL": "",
    "OWNER_ID": "",
    "UPSTREAM_REPO": "",
    "UPSTREAM_BRANCH": "",
    # Add any custom variables below:
    "CUSTOM_VAR_EXAMPLE": ""
}

def get_config_vars(app_name):
    resp = requests.get(f"https://api.heroku.com/apps/{app_name}/config-vars", headers=headers)
    return resp.json() if resp.status_code == 200 else {}

def update_config_vars(app_name, new_vars):
    # Filter out empty values
    vars_to_update = {k: v for k, v in new_vars.items() if v != ""}
    
    if not vars_to_update:
        print("No non-empty variables to update.")
        return
        
    print(f"Updating {len(vars_to_update)} config vars...")
    resp = requests.patch(f"https://api.heroku.com/apps/{app_name}/config-vars", headers=headers, json=vars_to_update)
    
    if resp.status_code == 200:
        print("✅ Config Vars updated successfully!")
    else:
        print(f"❌ Failed to update Config Vars: {resp.text}")

if app_info:
    current_vars = get_config_vars(APP_NAME)
    print(f"Current variables on Heroku: {len(current_vars)}")
    update_config_vars(APP_NAME, CONFIG_VARS)


In [ ]:
# @title Backup & Restore Config Vars
# @markdown Run this to save your current config vars to a file or load them.
ACTION = "Backup to JSON" # @param ["Backup to JSON", "Restore from JSON"]
FILE_NAME = "config_backup.json" # @param {type:"string"}

import json
import os

if app_info:
    if ACTION == "Backup to JSON":
        current_vars = get_config_vars(APP_NAME)
        with open(FILE_NAME, "w") as f:
            json.dump(current_vars, f, indent=4)
        print(f"✅ Config Vars backed up to {FILE_NAME}")
    
    elif ACTION == "Restore from JSON":
        if os.path.exists(FILE_NAME):
            with open(FILE_NAME, "r") as f:
                loaded_vars = json.load(f)
            update_config_vars(APP_NAME, loaded_vars)
        else:
            print(f"❌ File {FILE_NAME} not found.")


In [ ]:
# @title 6. Buildpacks
# @markdown Ensures `heroku/python` is configured.

def manage_buildpacks(app_name):
    print("Fetching current buildpacks...")
    resp = requests.get(f"https://api.heroku.com/apps/{app_name}/buildpack-installations", headers=headers)
    
    current_buildpacks = []
    if resp.status_code == 200:
        current_buildpacks = [bp['buildpack']['url'] for bp in resp.json()]
    
    has_python = any('heroku/python' in bp for bp in current_buildpacks)
    
    if has_python:
        print("✅ heroku/python buildpack is already installed.")
    else:
        print("Installing heroku/python buildpack...")
        payload = {
            "updates": [
                {"buildpack": "heroku/python"}
            ]
        }
        # If there are existing, we should probably append, but Heroku PUT replaces the list.
        # So let's construct the full list
        updates = [{"buildpack": bp} for bp in current_buildpacks]
        updates.append({"buildpack": "heroku/python"})
        
        resp = requests.put(f"https://api.heroku.com/apps/{app_name}/buildpack-installations", headers=headers, json={"updates": updates})
        
        if resp.status_code == 200:
            print("✅ Buildpack updated successfully!")
        else:
            print(f"❌ Failed to update buildpacks: {resp.text}")

if app_info:
    manage_buildpacks(APP_NAME)


In [ ]:
# @title 7. Deployment (GitHub to Heroku)
# @markdown Provide the GitHub repository URL and branch you wish to deploy.
REPO_URL = "https://github.com/yt-dlp/yt-dlp" # @param {type:"string"}
BRANCH = "master" # @param {type:"string"}

import time

def trigger_build(app_name, repo_url, branch):
    # Ensure URL ends properly
    clean_repo = repo_url.rstrip("/")
    if clean_repo.endswith(".git"):
        clean_repo = clean_repo[:-4]
        
    tarball_url = f"{clean_repo}/archive/refs/heads/{branch}.tar.gz"
    print(f"Source Tarball URL: {tarball_url}")
    
    payload = {
        "source_blob": {
            "url": tarball_url
        }
    }
    
    print("Triggering Heroku Build...")
    start_time = time.time()
    resp = requests.post(f"https://api.heroku.com/apps/{app_name}/builds", headers=headers, json=payload)
    
    if resp.status_code == 201:
        build_data = resp.json()
        print("✅ Build triggered successfully!")
        
        build_id = build_data['id']
        stream_url = build_data['output_stream_url']
        
        print("Streaming build logs...")
        print("-" * 40)
        
        # Stream logs
        try:
            with requests.get(stream_url, stream=True) as log_resp:
                for line in log_resp.iter_lines():
                    if line:
                        print(line.decode('utf-8'))
        except Exception as e:
            print(f"Error streaming logs: {e}")
            
        print("-" * 40)
        
        # Check final status
        status = "pending"
        while status in ["pending", "building"]:
            b_resp = requests.get(f"https://api.heroku.com/apps/{app_name}/builds/{build_id}", headers=headers)
            if b_resp.status_code == 200:
                status = b_resp.json()['status']
                if status in ["pending", "building"]:
                    time.sleep(5)
            else:
                break
                
        duration = round(time.time() - start_time, 2)
        if status == "succeeded":
            print(f"✅ Build completed successfully in {duration} seconds.")
        else:
            print(f"❌ Build failed with status: {status}")
            
    else:
        print(f"❌ Failed to trigger build: {resp.text}")

if app_info:
    trigger_build(APP_NAME, REPO_URL, BRANCH)


In [ ]:
# @title 8. Dyno Management
# @markdown Scale your dynos and set the tier.
DYNO_TIER = "eco" # @param ["eco", "basic", "standard-1X", "standard-2X", "performance-m", "performance-l"]
WEB_DYNOS = 1 # @param {type:"integer"}
WORKER_DYNOS = 1 # @param {type:"integer"}
ACTION = "Scale Dynos" # @param ["Scale Dynos", "Restart All Dynos"]

def scale_dynos(app_name):
    updates = []
    if WEB_DYNOS >= 0:
        updates.append({"process": "web", "quantity": WEB_DYNOS, "size": DYNO_TIER})
    if WORKER_DYNOS >= 0:
        updates.append({"process": "worker", "quantity": WORKER_DYNOS, "size": DYNO_TIER})
        
    if not updates:
        return
        
    print(f"Scaling dynos to Tier: {DYNO_TIER}...")
    resp = requests.patch(f"https://api.heroku.com/apps/{app_name}/formation", headers=headers, json={"updates": updates})
    
    if resp.status_code == 200:
        print("✅ Dynos scaled successfully!")
        for formation in resp.json():
            print(f"- {formation['type']}: {formation['quantity']} ({formation['size']})")
    else:
        print(f"❌ Failed to scale dynos: {resp.text}")

def restart_dynos(app_name):
    print("Restarting all dynos...")
    resp = requests.delete(f"https://api.heroku.com/apps/{app_name}/dynos", headers=headers)
    if resp.status_code == 202:
        print("✅ Restart signal sent successfully.")
    else:
        print(f"❌ Failed to restart dynos: {resp.text}")

if app_info:
    if ACTION == "Scale Dynos":
        scale_dynos(APP_NAME)
    elif ACTION == "Restart All Dynos":
        restart_dynos(APP_NAME)


In [ ]:
# @title 9. Deployment Status & Logs
# @markdown View app status and fetch recent logs.
FETCH_LOGS = True # @param {type:"boolean"}
LINES = 50 # @param {type:"integer"}

def show_status(app_info):
    app_name = app_info['name']
    print(f"=== Status for {app_name} ===")
    print(f"🌍 Web URL: {app_info.get('web_url', 'N/A')}")
    print(f"📦 Region: {app_info.get('region', {}).get('name', 'N/A')}")
    
    # Get releases
    resp = requests.get(f"https://api.heroku.com/apps/{app_name}/releases", headers=headers)
    if resp.status_code == 200:
        releases = resp.json()
        if releases:
            latest = releases[-1]
            print(f"🚀 Latest Release: v{latest.get('version')} - {latest.get('description')} ({latest.get('created_at')})")
            
    # Get Dyno status
    resp = requests.get(f"https://api.heroku.com/apps/{app_name}/dynos", headers=headers)
    if resp.status_code == 200:
        dynos = resp.json()
        print("
=== Running Dynos ===")
        if dynos:
            for dyno in dynos:
                print(f"- {dyno['name']} ({dyno['type']}): {dyno['state']}")
        else:
            print("No dynos running.")
            
    print("
🔗 Dashboard Links:")
    print(f"Dashboard: https://dashboard.heroku.com/apps/{app_name}")
    print(f"Logs: https://dashboard.heroku.com/apps/{app_name}/logs")

def fetch_logs(app_name, lines):
    print(f"
Fetching last {lines} lines of runtime logs...")
    payload = {"lines": lines, "source": "app"}
    resp = requests.post(f"https://api.heroku.com/apps/{app_name}/log-sessions", headers=headers, json=payload)
    
    if resp.status_code == 201:
        log_url = resp.json()['logplex_url']
        try:
            log_resp = requests.get(log_url)
            print("-" * 40)
            print(log_resp.text)
            print("-" * 40)
        except Exception as e:
            print(f"Error reading logs: {e}")
    else:
        print(f"❌ Failed to create log session: {resp.text}")

if app_info:
    show_status(app_info)
    if FETCH_LOGS:
        fetch_logs(APP_NAME, LINES)


In [ ]:
# @title 10. Teardown (Delete App)
# @markdown **DANGER ZONE:** Check the box below and run this cell to permanently delete the app.
CONFIRM_DELETE = False # @param {type:"boolean"}

def delete_app(app_name):
    if not CONFIRM_DELETE:
        print("⚠️ Deletion aborted. Check the CONFIRM_DELETE box to proceed.")
        return
        
    print(f"Attempting to delete app {app_name}...")
    resp = requests.delete(f"https://api.heroku.com/apps/{app_name}", headers=headers)
    
    if resp.status_code == 200:
        print(f"✅ App {app_name} successfully deleted.")
    else:
        print(f"❌ Failed to delete app: {resp.text}")

if app_info:
    delete_app(APP_NAME)
